# 01. Project: RAG over [MPhil ML Notes (CS-567 & CS-667)](https://github.com/S33mi/machine-learning-journey)

**Corpus:** Lecture notes from **Dr. Nazar Khan**, PUCIT  
**Courses:** CS-567 Machine Learning · CS-667 Advanced Machine Learning  

This notebook builds a study RAG system over **the extracted lecture notes (26 topics)**, not generic LLM curriculum text.

Pipeline:
1. Load structured notes (`id`, `title`, `course`, `text`)
2. Chunk → embed → FAISS
3. Retrieve top-k passages
4. Generate grounded answers (GPU-first / CPU fallback)
5. `ask()` helper for exam-style questions


## 1. Setup

```bash
pip install transformers sentence-transformers faiss-cpu langchain-text-splitters accelerate
```


In [13]:
# pip install transformers sentence-transformers faiss-cpu langchain-text-splitters accelerate

In [14]:
import json
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cpu


## 2. Load Your Lecture Notes

26 notes covering CS-567 (probability, curve fitting, MAP/MLE, density estimation, linear models, …) and CS-667 (PCA, GMM, EM, neural nets, CNN, SVM, boosting, spectral clustering, MDN, …).


In [15]:
from pathlib import Path

notes_path = Path("/content/cs567_cs667_notes.json") #/content/cs567_cs667_notes.json
# Fallback if running from another cwd
if not notes_path.exists():
    notes_path = Path("/home/workdir/artifacts/cs567_cs667_notes.json")

NOTES = json.loads(notes_path.read_text(encoding="utf-8"))
print(f"Loaded {len(NOTES)} notes")
for n in NOTES[:5]:
    print(f"  - [{n['course']}] {n['title']}")
print("  ...")
for n in NOTES[-3]:
    pass
for n in NOTES[-3:]:
    print(f"  - [{n['course']}] {n['title']}")


Loaded 26 notes
  - [CS-567] Introduction to Machine Learning
  - [CS-567] Curve Fitting and Regularisation
  - [CS-567] Probability Theory
  - [CS-567] Maximum Likelihood Estimation (Probabilistic Curve Fitting)
  - [CS-567] Maximum A-Posteriori Estimation (Bayesian Curve Fitting)
  ...
  - [CS-667] Conditional Mixture Models
  - [CS-667] Mixture Density Networks
  - [CS-667] Spectral Clustering


## 3. Chunk → Embed → Index


In [16]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for note in NOTES:
    parts = splitter.split_text(note["text"])
    for i, part in enumerate(parts):
        chunks.append({
            "chunk_id": f"{note['id']}_{i}",
            "note_id": note["id"],
            "title": note["title"],
            "course": note.get("course", ""),
            "source_file": note.get("source_file", ""),
            "text": part.strip(),
        })

print(f"Chunks: {len(chunks)}")

EMBED_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_NAME, device=DEVICE)
texts = [c["text"] for c in chunks]
emb = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
emb = np.asarray(emb, dtype="float32")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print(f"FAISS: {index.ntotal} vectors, dim={emb.shape[1]}")


Chunks: 84


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

FAISS: 84 vectors, dim=384


## 4. Retriever


In [17]:
def retrieve(query: str, k: int = 4):
    q = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.asarray(q, dtype="float32"), k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        c = chunks[int(idx)]
        hits.append({
            "score": float(score),
            "title": c["title"],
            "course": c["course"],
            "text": c["text"],
            "chunk_id": c["chunk_id"],
            "note_id": c["note_id"],
            "source_file": c["source_file"],
        })
    return hits


for h in retrieve("What is the EM algorithm?", k=3):
    print(f"[{h['score']:.3f}] {h['course']} | {h['title']}")
    print(f"   {h['text'][:100]}...\n")


[0.623] CS-667 | The EM Algorithm
   . Each iteration is guaranteed not to decrease the observed-data likelihood. MAP estimation is obtai...

[0.547] CS-667 | K-means Clustering
   K-means partitions data into K clusters by minimising the sum of squared distances between points an...

[0.428] CS-667 | The EM Algorithm
   Algorithms: 1. Initialise θ_old. 2. E-step: evaluate p(Z|X,θ_old). 3. M-step: θ_new = arg max_θ Q(θ,...



## 5. Generator (Transformers v5 compatible)

Uses `AutoModelForSeq2SeqLM.generate()` — not the removed `text2text-generation` pipeline.


In [18]:
GEN_NAME = "google/flan-t5-large" if DEVICE == "cuda" else "google/flan-t5-base" #google/flan-t5-small
gen_tok = AutoTokenizer.from_pretrained(GEN_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_NAME).to(DEVICE)
gen_model.eval()
print(f"Generator: {GEN_NAME}")


def generate(prompt: str, max_new_tokens: int = 140) -> str:
    inputs = gen_tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return gen_tok.decode(out[0], skip_special_tokens=True).strip()


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator: google/flan-t5-base


## 6. Grounded `ask()` for exam-style questions


In [19]:
def build_prompt(question: str, hits: list) -> str:
    ctx = "\n\n".join(
        f"[{i}] ({h['course']} – {h['title']}) {h['text']}"
        for i, h in enumerate(hits, 1)
    )
    return (
        "You are a study assistant for CS-567 / CS-667 (Dr. Nazar Khan). "
        "Answer ONLY using the lecture notes below. "
        "If the notes do not contain the answer, say "
        "\"I don't know based on the provided notes.\" "
        "Be concise and exam-oriented.\n\n"
        f"Notes:\n{ctx}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def ask(question: str, k: int = 4, show_sources: bool = True) -> str:
    hits = retrieve(question, k=k)
    if show_sources:
        print("Sources:")
        for h in hits:
            print(f"  [{h['score']:.3f}] {h['course']} | {h['title']} ({h['source_file']})")
        print()
    return generate(build_prompt(question, hits))


print("ask() ready.")


ask() ready.


## 7. Study / exam questions


In [20]:
questions = [
    "What is over-fitting and how does regularisation help?",
    "Explain the equivalence of MAP estimation and regularised least-squares.",
    "What are the two steps of the EM algorithm?",
    "Why does a single Gaussian fail on multi-modal data?",
    "What is the kernel trick in SVMs?",
    "How does spectral clustering differ from k-means?",
    "What does a Mixture Density Network output and why is it useful?",
    "Why do CNNs use weight sharing and local connectivity?",
    "Who invented the telephone?",  # out of notes
]

for q in questions:
    print("=" * 60)
    print(f"Q: {q}")
    print(f"A: {ask(q)}\n")


Q: What is over-fitting and how does regularisation help?
Sources:
  [0.681] CS-567 | Curve Fitting and Regularisation (lecture_02_03_curve_fitting.pdf)
  [0.499] CS-567 | Curve Fitting and Regularisation (lecture_02_03_curve_fitting.pdf)
  [0.422] CS-567 | Maximum A-Posteriori Estimation (Bayesian Curve Fitting) (lecture_06_maximum_posterior_estimation.pdf)
  [0.333] CS-567 | Maximum A-Posteriori Estimation (Bayesian Curve Fitting) (lecture_06_maximum_posterior_estimation.pdf)

A: I don't know based on the provided notes.

Q: Explain the equivalence of MAP estimation and regularised least-squares.
Sources:
  [0.590] CS-567 | Maximum A-Posteriori Estimation (Bayesian Curve Fitting) (lecture_06_maximum_posterior_estimation.pdf)
  [0.566] CS-567 | Maximum A-Posteriori Estimation (Bayesian Curve Fitting) (lecture_06_maximum_posterior_estimation.pdf)
  [0.520] CS-567 | Maximum Likelihood Estimation (Probabilistic Curve Fitting) (lecture_05_maximum_likelihood_estimation.pdf)
  [0.452] CS-66

## 8. Filter by course (optional)


In [21]:
def retrieve_course(query: str, course: str, k: int = 4):
    """Retrieve then keep only chunks from the given course code."""
    hits = retrieve(query, k=max(k * 3, 10))
    filtered = [h for h in hits if h["course"].upper() == course.upper()]
    return filtered[:k]


print("CS-667 only – EM:")
for h in retrieve_course("EM algorithm", "CS-667", k=3):
    print(f"  {h['title']}: {h['text'][:80]}...")


CS-667 only – EM:
  The EM Algorithm: . Each iteration is guaranteed not to decrease the observed-data likelihood. MAP...
  K-means Clustering: K-means partitions data into K clusters by minimising the sum of squared distanc...
  The EM Algorithm: Algorithms: 1. Initialise θ_old. 2. E-step: evaluate p(Z|X,θ_old). 3. M-step: θ_...


## 9. Persist index for reuse


In [22]:
from pathlib import Path
import pickle

save_dir = Path("./cs567_cs667_rag")
save_dir.mkdir(exist_ok=True)
faiss.write_index(index, str(save_dir / "notes.faiss"))
with open(save_dir / "chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)
print("Saved to", save_dir.resolve())


Saved to /content/cs567_cs667_rag


## 10. Summary

| Item | Value |
|------|--------|
| **Notes** | 26 lecture topics (CS-567 + CS-667) |
| **Source** | Extracted notes from Dr. Nazar Khan lecture PDFs **(excluding the personal hand written & other learning notes)** |
| **Embeddings** | `all-MiniLM-L6-v2` |
| **Index** | FAISS inner product |
| **Generator** | FLAN-T5 (base on GPU, small on CPU) |
| **API** | `ask(question)` with source citations |

Related implementations: [S33mi/machine-learning-journey](https://github.com/S33mi/machine-learning-journey)

---

**Alternative project LLM notebook** [`01_rag_over_ml2llm_notes.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/06_projects/01_rag_over_ml2llm_notes.ipynb)

**Next project notebook:** [`02_hybrid_classical_plus_llm.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/06_projects/02_hybrid_classical_plus_llm.ipynb)
Combining Classical ML (PCA, GMM, etc.) with LLMs

---

**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Science/Analytics and ML/AI related opportunities

---